# High-confidence BBCH consolidation checks

## tl;dr

The consolidated CSV retains only valid, direct source BBCH codes. Inferred mappings and unmapped phases are excluded before exact seven-field deduplication. Source-qualified identifiers do not establish physical-site independence.

## Context & Methods

Independent read-only checks of `phenology_data.csv`, its source-count report and compressed row-level provenance. Preparation is reproducible with `organize_phenology.py`.

### Key Assumptions

All countries are eligible, but only directly coded BBCH records dated 1951-01-01 or later qualify. Chronology-v2 additionally screens seasonal timing, stage order and phase intervals. Quarantined observations are not proven errors; uncommon management and genuine extremes can fail screening. Cross-source station matching and inference of missing varieties remain outside consolidation.


## Data

Read the local CSV as strings to preserve BBCH leading zeros. Only Python's standard library is required.


In [1]:
import csv, gzip, hashlib, json, math, re
from collections import Counter
from datetime import date
from pathlib import Path
import subprocess, sys
root = next(p for p in [Path('data/phenology'), Path('.')] if (p / 'phenology_data.csv').is_file())
with (root / 'phenology_data.csv').open(encoding='utf-8', newline='') as stream:
    reader = csv.DictReader(stream)
    columns = reader.fieldnames
    records = [tuple(row[column] for column in columns) for row in reader]
quality = json.loads((root / 'phenology_data_quality.json').read_text())
assert columns == ['site', 'variety', 'country', 'lat', 'lon', 'Date', 'BBCH']
assert len(records) == quality['output_rows'] == len(set(records))
assert hashlib.sha256((root / 'phenology_data.csv').read_bytes()).hexdigest() == quality['output_sha256']
print(f'CSV rows: {len(records):,}; exact duplicates: 0; schema and SHA256 verified.')


CSV rows: 259,998; exact duplicates: 0; schema and SHA256 verified.


## Results

### Validity and completeness


In [2]:
missing = Counter()
countries = Counter()
sites = set()
for record in records:
    site, variety, country, latitude, longitude, event_date, bbch = record
    assert site and (not country or re.fullmatch('[A-Z]{2}', country))
    assert country == site.split(':')[0] or (not country and site.startswith('XX:'))
    assert latitude and longitude and -90 <= float(latitude) <= 90 and -180 <= float(longitude) <= 180
    assert date.fromisoformat(event_date).isoformat() == event_date
    assert event_date >= '1951-01-01'
    assert re.fullmatch('[0-9]{2}', bbch)
    missing.update(column for column, value in zip(columns, record) if not value)
    countries[country or 'unknown'] += 1
    sites.add(site)
assert len(sites) == quality['source_qualified_sites']
assert dict(missing) == {k: v for k, v in quality['missing_fields'].items() if v}
assert dict(countries) == {k: v['rows'] for k, v in quality['countries'].items()}
print(f'Source-qualified sites: {len(sites):,}; identified countries: {len(countries)-int("unknown" in countries)}.')
print('Missing fields:', dict(missing))
print('Date coverage:', min(r[5] for r in records), 'to', max(r[5] for r in records))


Source-qualified sites: 4,244; identified countries: 13.
Missing fields: {'variety': 176904}
Date coverage: 1951-03-14 to 2026-06-24


### Source reconciliation and row-level provenance

Every included or duplicate source row must point to an actual exported event hash. Every exported event must be represented exactly once as retained in the provenance.


In [3]:
event_hashes = {hashlib.sha256(json.dumps(record, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest() for record in records}
retained_hashes = set()
qc_hashes = set()
statuses = Counter()
per_source = Counter()
with gzip.open(root / 'phenology_data_provenance.jsonl.gz', 'rt', encoding='utf-8') as stream:
    for line in stream:
        item = json.loads(line)
        statuses[item['status']] += 1
        per_source[(item['source'], item['status'])] += 1
        if item['status'] == 'excluded_chronology_qc':
            qc_hashes.add(item['candidate_event_id'])
        if item['status'] in {'retained', 'duplicate'}:
            assert item['event_id'] in event_hashes
            assert item['bbch_mapping'] == 'direct'
        if item['status'] == 'retained':
            assert item['event_id'] not in retained_hashes
            retained_hashes.add(item['event_id'])
assert retained_hashes == event_hashes
assert sum(statuses.values()) == sum(s['input_rows'] for s in quality['inputs'])
for source in quality['inputs']:
    assert per_source[(source['path'], 'retained')] == source['output_rows']
    assert per_source[(source['path'], 'duplicate')] == source['duplicate_rows']
    assert per_source[(source['path'], 'excluded_non_grape')] == source['excluded_non_grape']
    assert per_source[(source['path'], 'excluded_inferred_bbch')] == source['excluded_inferred_bbch']
    assert per_source[(source['path'], 'excluded_unmapped_bbch')] == source['excluded_unmapped_bbch']
    assert per_source[(source['path'], 'excluded_before_1951')] == source['excluded_before_1951']
    assert per_source[(source['path'], 'excluded_missing_or_invalid_date')] == source['excluded_missing_or_invalid_date']
    assert per_source[(source['path'], 'excluded_chronology_qc')] == source['excluded_chronology_qc']
assert hashlib.sha256((root / 'phenology_data_provenance.jsonl.gz').read_bytes()).hexdigest() == quality['provenance_sha256']
print('Provenance records:', dict(statuses))
print('All exported rows trace to source records; input and output counts reconcile.')


Provenance records: {'excluded_before_1951': 17494, 'retained': 259998, 'duplicate': 394045, 'excluded_chronology_qc': 32466, 'excluded_non_grape': 25471, 'excluded_inferred_bbch': 79864, 'excluded_unmapped_bbch': 13690}
All exported rows trace to source records; input and output counts reconcile.


### Independent chronology and quarantine verification
Literal screening bounds below do not import the filtering implementation.


In [4]:
from collections import defaultdict
from itertools import combinations
stage_groups = defaultdict(lambda: defaultdict(list))
for row in records:
    b = int(row[6]); d = date.fromisoformat(row[5]); md = row[5][5:]
    limits = ('02-01','06-15') if 1<=b<=9 else ('03-01','07-31') if 10<=b<=19 else ('04-01','07-15') if 50<=b<=59 else ('05-01','07-31') if 60<=b<=69 else ('05-15','09-15') if 70<=b<=79 else ('07-01','10-15') if 80<=b<=85 else ('07-15','11-15') if 86<=b<=89 else None
    assert limits is None or limits[0] <= md <= limits[1]
    stage_groups[(row[0],row[1],d.year)][b].append(d.toordinal())
branches = [[1,3,5,7,9,10,11,12,13,14,15,16,17,18,19], [1,3,5,7,9,53,55,57,60,61,62,63,64,65,66,67,68,69,71,73,75,77,79,81,83,85,87,89]]
pairs = set(pair for branch in branches for pair in combinations(branch,2))
bounds = {(7,65):(20,100),(9,65):(20,100),(65,81):(30,120),(81,89):(10,90)}
for stages in stage_groups.values():
    for a,b in pairs:
        if a not in stages or b not in stages: continue
        low,high = bounds.get((a,b),(0,366))
        assert all(any(low<=y-x<=high for y in stages[b]) for x in stages[a])
        assert all(any(low<=y-x<=high for x in stages[a]) for y in stages[b])
with gzip.open(root / 'phenology_data_quarantine.jsonl.gz','rt',encoding='utf-8') as stream:
    quarantined = [json.loads(line) for line in stream]
assert {q['event_id'] for q in quarantined} == qc_hashes
assert qc_hashes.isdisjoint(event_hashes)
excluded_rows = [tuple(q['record'][c] for c in columns) for q in quarantined]
for q,row in zip(quarantined,excluded_rows):
    assert q['evidence'] and q['policy'] == 'chronology-v2'
    assert hashlib.sha256(json.dumps(row,ensure_ascii=False,separators=(',',':')).encode()).hexdigest() == q['event_id']
assert len(quarantined) == len(qc_hashes) == quality['chronology_qc']['rows_quarantined']
restored = sorted(records+excluded_rows,key=lambda r:(r[2],r[0],r[5],r[6],r[1],r[3],r[4]))
assert len(restored) == len(set(restored)) == quality['chronology_qc']['rows_before']
assert hashlib.sha256(json.dumps(restored,ensure_ascii=False,separators=(',',':')).encode()).hexdigest() == quality['chronology_qc']['pre_qc_rows_sha256']
assert hashlib.sha256((root/'phenology_data_quarantine.jsonl.gz').read_bytes()).hexdigest() == quality['chronology_qc']['quarantine_sha256']
print(f'All retained rows pass independent seasonal and pair-support checks; {len(quarantined):,} quarantined rows reconstruct the complete pre-QC inventory without loss.')


All retained rows pass independent seasonal and pair-support checks; 11,034 quarantined rows reconstruct the complete pre-QC inventory without loss.


### Transformation edge cases


In [5]:
result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(root), '-p', 'test_*.py'], capture_output=True, text=True)
assert result.returncode == 0, result.stdout + result.stderr
print(result.stderr.strip())


..................
----------------------------------------------------------------------
Ran 18 tests in 0.002s

OK


## Takeaways

Every retained row has a direct source BBCH code; inferred conversions and unmapped phases are accounted for as exclusions. Missing varieties, maturity-group labels and potential DWD/PEP725 overlaps still require attention before modelling. Detailed methods and attribution are in `phenology_data.md`.
